# Layer 1 v2 Drilldown

Three panels backed by the v2 schema (canonical_universe + raw_* +
fetch_watermarks). Read-only. Run top-to-bottom.

**Panels:**
1. **Source Status** -- per-(source, field) watermark roll-up
2. **Provenance Summary** -- per-raw_table row counts + coverage
3. **Per-Ticker Drilldown** -- every raw_* slice for a given ticker

**Control Variables:**
- `ENABLED_SOURCES` -- override `config/datasources.yaml > enabled_sources` for this run
- `FORCE_REFRESH_SOURCES` -- bypass watermark short-circuits for the named sources
- `DRILLDOWN_TICKER` -- which ticker the drilldown panel shows

## Setup

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display, Markdown

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'src').exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.common.database import DatabaseManager
from src.common.datasources.registry import DataSourceRegistry
from src.layer1_universe.notebook_panels import (
    apply_notebook_overrides,
    per_ticker_drilldown,
    provenance_summary,
    source_status,
)

## Control Variables

In [ ]:
ENABLED_SOURCES = None
FORCE_REFRESH_SOURCES = []
DRILLDOWN_TICKER = 'AAPL'
DB_PATH = REPO_ROOT / 'data' / 'fundamentals.db'

In [ ]:
db = DatabaseManager(db_path=str(DB_PATH))
registry = DataSourceRegistry()
base_config = {'enabled_sources': [s.name for s in registry.enabled_sources()]}
effective_config = apply_notebook_overrides(
    base_config,
    enabled_sources=ENABLED_SOURCES,
    force_refresh_sources=FORCE_REFRESH_SOURCES,
)
display(Markdown(f'**Effective config for this run:** `{effective_config}`'))

## Panel 1 -- Source Status

In [ ]:
src_status = source_status(db, registry=registry)
display(src_status)

## Panel 2 -- Provenance Summary

In [ ]:
prov = provenance_summary(db)
display(prov)

## Panel 3 -- Per-Ticker Drilldown

In [ ]:
drilldown = per_ticker_drilldown(db, DRILLDOWN_TICKER)
for key, df in drilldown.items():
    display(Markdown(f'### {key} ({len(df)} rows)'))
    if len(df) > 0:
        display(df.head(20))
    else:
        display(Markdown('_no data_'))